# run the full **pipeline**

In [1]:
%cd /content
!rm -rf banking-data-platform
!git clone https://github.com/srk2695/banking-data-platform.git
%cd banking-data-platform
!pip install -q faker pyspark
!python src/generate_data.py
!python src/bronze_to_silver.py
!python src/silver_to_gold.py

/content
Cloning into 'banking-data-platform'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 69 (delta 22), reused 53 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 125.01 KiB | 1.84 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/banking-data-platform
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 7.2 MB/s eta 0:00:00
/content/banking-data-platform/src/generate_data.py:6: SyntaxWarning: invalid escape sequence '\g'
  python src\generate_data.py
Wrote to /content/banking-data-platform/data/bronze
  customers.csv        1,010 rows
  accounts.csv         1,960 rows
  transactions.csv    20,200 rows
Injected issues: nulls, duplicates, negative amounts, casing, orphan account IDs.
[0.028s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.0

# load the gold and silver **outputs**

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()
g = lambda n: spark.read.parquet(f"data/gold/{n}")
s = lambda n: spark.read.parquet(f"data/silver/{n}")

dim_customer, dim_account, fact = g("dim_customer"), g("dim_account"), g("fact_transactions")
daily, cust_sum = g("daily_txn_summary"), g("customer_summary")
silver_txn = s("transactions")

for name, df in [("dim_customer", dim_customer), ("dim_account", dim_account),
                 ("fact_transactions", fact), ("daily_txn_summary", daily),
                 ("customer_summary", cust_sum)]:
    print(f"{name:20s}{df.count():>8,}")

dim_customer           1,000
dim_account            1,960
fact_transactions     19,740
daily_txn_summary      1,093
customer_summary       1,000


# **validate the gold layer**

In [3]:
# 1. No rows lost between silver and the fact table
print("silver txns:", silver_txn.count(), "| fact txns:", fact.count())

# 2. Keys are unique
print("dup txn_key:     ", fact.groupBy("txn_key").count().filter("count > 1").count())
print("dup account_key: ", dim_account.groupBy("account_key").count().filter("count > 1").count())
print("dup customer_key:", dim_customer.groupBy("customer_key").count().filter("count > 1").count())

# 3. No orphan keys in the fact table
print("fact rows with no account:",
      fact.join(dim_account, "account_key", "left_anti").count())
print("fact rows with no customer:",
      fact.join(dim_customer, "customer_key", "left_anti").count())

# 4. Summaries reconcile to the fact table
print("daily summary txns:   ", daily.agg(F.sum("txn_count")).first()[0])
print("customer summary txns:", cust_sum.agg(F.sum("txn_count")).first()[0])
print("customer summary accts:", cust_sum.agg(F.sum("account_count")).first()[0])

# 5. Completed amounts agree: fact vs daily summary
fact_total = fact.filter("status = 'COMPLETED'").agg(F.round(F.sum("amount"), 2)).first()[0]
daily_total = daily.agg(F.round(F.sum("completed_amount"), 2)).first()[0]
print("completed amount, fact vs daily:", fact_total, daily_total)

silver txns: 19740 | fact txns: 19740
dup txn_key:      0
dup account_key:  0
dup customer_key: 0
fact rows with no account: 0
fact rows with no customer: 0
daily summary txns:    19740
customer summary txns: 19740
customer summary accts: 1960
completed amount, fact vs daily: 3019240.0 3019240.0


Silver txns equals fact txns (19,740).

Every duplicate-key count is 0.

Both orphan checks are 0.

The daily and customer summary transaction totals both equal 19,740, and the account total is 1,960.

The two completed-amount totals match. They can differ by a few cents because of rounding, but a big gap means a bug.

In [4]:
cust_sum.orderBy(F.col("aed_amount").desc()).select(
    "full_name", "segment", "account_count", "txn_count", "aed_amount").show(10)

daily.orderBy(F.col("txn_date").desc()).show(10)

+-----------------+-------+-------------+---------+----------+
|        full_name|segment|account_count|txn_count|aed_amount|
+-----------------+-------+-------------+---------+----------+
|      Susan Baker|PREMIUM|            3|       42|   14122.8|
|Samantha Gonzales| RETAIL|            3|       36|   10255.6|
|    Allen Kennedy|    SME|            2|       17|    9807.1|
|    Philip Wilson| RETAIL|            3|       30|   8058.94|
|    Megan Parsons|PREMIUM|            3|       41|   7657.98|
|      Roy Hawkins| RETAIL|            3|       32|    7650.1|
|   Mark Rodriguez|PREMIUM|            3|       39|   7407.42|
|  Ryan Richardson| RETAIL|            3|       29|   7178.05|
|      James White| RETAIL|            3|       39|   7151.13|
|  Stephanie Smith| RETAIL|            3|       34|   7039.47|
+-----------------+-------+-------------+---------+----------+
only showing top 10 rows
+----------+--------+---------+---------------+------------+----------------+----------+
|  t